In [1]:
!pip install datasets==3.6.0
!pip install sentence-transformers chromadb
!pip install ollama

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [2]:
import datasets
from datasets import load_dataset
import re
from sentence_transformers import SentenceTransformer
import ollama
import chromadb
from chromadb.config import Settings

In [3]:
# load the data
medical_ner_data = load_dataset("singh-aditya/MACCROBAT_biomedical_ner",trust_remote_code=True)
print(medical_ner_data)

DatasetDict({
    train: Dataset({
        features: ['full_text', 'ner_info', 'tokens', 'ner_labels'],
        num_rows: 200
    })
})


In [4]:
print(medical_ner_data["train"][0])

{'full_text': "CASE: A 28-year-old previously healthy man presented with a 6-week history of palpitations.\nThe symptoms occurred during rest, 2–3 times per week, lasted up to 30 minutes at a time and were associated with dyspnea.\nExcept for a grade 2/6 holosystolic tricuspid regurgitation murmur (best heard at the left sternal border with inspiratory accentuation), physical examination yielded unremarkable findings.\nAn electrocardiogram (ECG) revealed normal sinus rhythm and a Wolff– Parkinson– White pre-excitation pattern (Fig.1: Top), produced by a right-sided accessory pathway.\nTransthoracic echocardiography demonstrated the presence of Ebstein's anomaly of the tricuspid valve, with apical displacement of the valve and formation of an “atrialized” right ventricle (a functional unit between the right atrium and the inlet [inflow] portion of the right ventricle) (Fig.2).\nThe anterior tricuspid valve leaflet was elongated (Fig.2C, arrow), whereas the septal leaflet was rudimentary

In [5]:
#prepare docs
documents = []

for i, example in enumerate(medical_ner_data["train"]):

    documents.append({
        "id": f"case_{i}",

        "text": example["full_text"],

        "entities": example["ner_info"]
    })

print(documents[0])

{'id': 'case_0', 'text': "CASE: A 28-year-old previously healthy man presented with a 6-week history of palpitations.\nThe symptoms occurred during rest, 2–3 times per week, lasted up to 30 minutes at a time and were associated with dyspnea.\nExcept for a grade 2/6 holosystolic tricuspid regurgitation murmur (best heard at the left sternal border with inspiratory accentuation), physical examination yielded unremarkable findings.\nAn electrocardiogram (ECG) revealed normal sinus rhythm and a Wolff– Parkinson– White pre-excitation pattern (Fig.1: Top), produced by a right-sided accessory pathway.\nTransthoracic echocardiography demonstrated the presence of Ebstein's anomaly of the tricuspid valve, with apical displacement of the valve and formation of an “atrialized” right ventricle (a functional unit between the right atrium and the inlet [inflow] portion of the right ventricle) (Fig.2).\nThe anterior tricuspid valve leaflet was elongated (Fig.2C, arrow), whereas the septal leaflet was 

In [6]:
#Chunk documents
import re

def chunk_document(document, max_chars=1200):

    text = document["text"]
    entities = document["entities"]

    # Approximate sentence segmentation
    sentences = [
        match for match in re.finditer(
            r'.+?(?:[.!?](?=\s|$)|$)',
            text,
            re.DOTALL
        )
        if match.group().strip()
    ]

    chunks = []
    current_sentences = []
    current_length = 0

    for sentence in sentences:

        sentence_length = len(sentence.group())

        if current_sentences and current_length + sentence_length > max_chars:

            start = current_sentences[0].start()
            end = current_sentences[-1].end()

            chunks.append((start, end))

            # Keep one sentence of overlap
            current_sentences = current_sentences[-1:]
            current_length = len(
                current_sentences[0].group()
            )

        current_sentences.append(sentence)
        current_length += sentence_length

    if current_sentences:

        chunks.append((
            current_sentences[0].start(),
            current_sentences[-1].end()
        ))

    output = []

    for chunk_index, (start, end) in enumerate(chunks):

        chunk_text = text[start:end]

        # Find entities that overlap the chunk
        chunk_entities = [
            entity for entity in entities
            if entity["start"] >= start
            and entity["end"] <= end
        ]

        output.append({

            "id": f"{document['id']}_chunk_{chunk_index}",

            "case_id": document["id"],

            "text": chunk_text,

            "entities": chunk_entities

        })

    return output

In [7]:

#Prepare all documents

all_chunks = []

for document in documents:

    chunks = chunk_document(document)

    all_chunks.extend(chunks)

print("Total chunks:", len(all_chunks))
print(all_chunks[0])


Total chunks: 629
{'id': 'case_0_chunk_0', 'case_id': 'case_0', 'text': "CASE: A 28-year-old previously healthy man presented with a 6-week history of palpitations.\nThe symptoms occurred during rest, 2–3 times per week, lasted up to 30 minutes at a time and were associated with dyspnea.\nExcept for a grade 2/6 holosystolic tricuspid regurgitation murmur (best heard at the left sternal border with inspiratory accentuation), physical examination yielded unremarkable findings.\nAn electrocardiogram (ECG) revealed normal sinus rhythm and a Wolff– Parkinson– White pre-excitation pattern (Fig.1: Top), produced by a right-sided accessory pathway.\nTransthoracic echocardiography demonstrated the presence of Ebstein's anomaly of the tricuspid valve, with apical displacement of the valve and formation of an “atrialized” right ventricle (a functional unit between the right atrium and the inlet [inflow] portion of the right ventricle) (Fig.2).\nThe anterior tricuspid valve leaflet was elongated (

In [8]:
print(all_chunks[2])

{'id': 'case_1_chunk_0', 'case_id': 'case_1', 'text': 'The patient was a 34-yr-old man who presented with complaints of fever and a chronic cough.\nHe was a smoker and had a history of pulmonary tuberculosis that had been treated and cured.\nA computed tomographic (CT) scan revealed multiple tiny nodules in both lungs.\nA thoracoscopic lung biopsy was taken from the right upper lobe.\nThe microscopic examination revealed a typical LCH.\nThe tumor cells had vesicular and grooved nuclei, and they formed small aggregations around the bronchioles (Fig.1).\nThe tumor cells were strongly positive for S-100 protein, vimentin, CD68 and CD1a.\nThere were infiltrations of lymphocytes and eosinophils around the tumor cells.\nWith performing additional radiologic examinations, no other organs were thought to be involved.\nHe quit smoking, but he received no other specific treatment.\nHe was well for the following one year.\nAfter this, a follow-up CT scan was performed and it showed a 4 cm-sized m

In [9]:
#create embeddings
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

#Embed all chunks
texts = [
    chunk["text"]
    for chunk in all_chunks
]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

In [10]:
#Store in chromadb
import chromadb

client = chromadb.PersistentClient(
    path="./medical_rag_db",
    settings = Settings(allow_reset=True)
)
client.reset()

collection = client.get_or_create_collection(
    name="maccrobat_documents",
    metadata={"hnsw:space": "cosine"}
)

#Add chunks
for i, chunk in enumerate(all_chunks):

    entity_names = [
        entity["text"]
        for entity in chunk["entities"]
    ]

    entity_types = [
        entity["label"]
        for entity in chunk["entities"]
    ]

    collection.add(

        ids=[chunk["id"]],

        documents=[chunk["text"]],

        embeddings=[embeddings[i].tolist()],

        metadatas=[{

            "case_id": chunk["case_id"],

            "entities": ", ".join(entity_names),

            "entity_types": ", ".join(entity_types)

        }]
    )


In [11]:
#Retrieve relevant chunks based on question
def retrieve_chunks(query):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5,
        include=["documents", "metadatas", "distances"]
    )
    return results

In [12]:
query = "What were the tumor marker test results of the old man with a retroperitoneal metastatic lymph node?"

results = retrieve_chunks(query)
for i, doc in enumerate(results["documents"][0]):

    print(f"\n--- Retrieved Chunk {i+1} ---")

    print("Text:", doc)

    print("Metadata:", results["metadatas"][0][i])

    print("Distance:", results["distances"][0][i])


--- Retrieved Chunk 1 ---
Text: 
Based on the tumor location, the clinical diagnosis was solitary metastasis to a mesocolic LN or HCC dissemination.
Determining the ideal therapeutic strategy for solitary but extrahepatic rare metastasis was difficult.
Although rapid growth was a critical concern in this case, the tumor was solitary and not accompanied by other metastases.
Considering both diagnostic and therapeutic viewpoints, we finally chose surgical resection in this case.
No disseminative nodules, lymphadenopathy, or ascites was observed during surgery.
The tumor was located in the mesocolon nearly at the wall of the descending colon, and partial resection of the descending colon with regional mesocolon was performed.
The patient’s postoperative course was uneventful, and she was discharged on postoperative day 8.
Serum levels of tumor marker decreased immediately after surgery (Figure 1).
Macroscopically, the mesocolic tumor was a solid and elastic mass with a smooth surface (Fi

In [13]:
#Function to create an instruction prompt with the required context from results
def create_rag_prompt(results):

    context = ""

    for i, document in enumerate(
        results["documents"][0]
    ):

        metadata = results["metadatas"][0][i]

        context += f"""
        Source: {metadata['case_id']}

        Medical document:
        {document}

        Medical entities:
        {metadata['entities']}

        ---
        """

    prompt = f"""
    You are an assistant that answers questions
    about a collection of medical case reports.

    Answer the user's question using only the
    information contained in the provided documents.

    If the documents do not contain sufficient
    information, explicitly state that.

    Do not invent medical information under any circumstances.

    Context:
    {context}
    """

    return prompt


In [14]:
query = "What were the tumor marker test results of the old man with a retroperitoneal metastatic lymph node?"

results = retrieve_chunks(query)

In [15]:
#Create the prompt that will be passed to the LLM
prompt = create_rag_prompt(results)
print(prompt)


    You are an assistant that answers questions
    about a collection of medical case reports.

    Answer the user's question using only the
    information contained in the provided documents.

    If the documents do not contain sufficient
    information, explicitly state that.

    Do not invent medical information under any circumstances.

    Context:
    
        Source: case_134

        Medical document:
        
Based on the tumor location, the clinical diagnosis was solitary metastasis to a mesocolic LN or HCC dissemination.
Determining the ideal therapeutic strategy for solitary but extrahepatic rare metastasis was difficult.
Although rapid growth was a critical concern in this case, the tumor was solitary and not accompanied by other metastases.
Considering both diagnostic and therapeutic viewpoints, we finally chose surgical resection in this case.
No disseminative nodules, lymphadenopathy, or ascites was observed during surgery.
The tumor was located in the mesocolon 

In [16]:
!pip install ollama

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


In [17]:
import ollama
language_model = 'llama3.1'

stream = ollama.chat(
    model = language_model,
    messages=[
        {'role': 'system', 'content': prompt},
        {'role': 'user', 'content': query}
    ],
    stream = True,
)

print("Chatbot response:")
for part in stream:
    print(part['message']['content'], end='', flush=True)

Chatbot response:
According to case_194, the tumor marker test results for the 60-year-old man with a retroperitoneal metastatic lymph node were:

* Carcinoembryonic antigen (CEA): 3.7 ng/mL
* Alpha fetoprotein (AFP): 17.3 ng/mL
* Carbohydrate antigen 19-9 (CA 19-9): 6.1 U/mL

All of these results were reported as normal.

In [18]:
query = "For the 28 year old man with heart palpitations, how long each one last?"
results = retrieve_chunks(query)
prompt = create_rag_prompt(results)
language_model = 'llama3.1'

stream = ollama.chat(
    model = language_model,
    messages=[
        {'role': 'system', 'content': prompt},
        {'role': 'user', 'content': query}
    ],
    stream = True,
)

print("Chatbot response:")
for part in stream:
    print(part['message']['content'], end='', flush=True)

Chatbot response:
According to the case report (case_0), the heart palpitations in the 28-year-old man last up to 30 minutes at a time.